In [1]:
import method as mth
import importlib, time
importlib.reload(mth)
from method import *

Generate Circuit

In [2]:
n, d, k = 30, 8, 7
print("qubit number is", n, ",depth is", d, ",T-gate number is", k)

#Generate random local circuit
circ = random_local_1d_clifford_plus_kT(n, d, k, seed=1235)

print(circ)
print("Total T gates:", sum(1 for op in circ.all_operations() if op.gate == cirq.T))

qubit number is 30 ,depth is 8 ,T-gate number is 7
0: ────S^-1───@───S─────────────────@───H──────────H──────────H──────T───S──────────H──────────
              │                     │
1: ────S^-1───X───H──────×───H──────@───S──────@───S^-1───────S^-1───────S^-1───────S──────@───
                         │                     │                                           │
2: ────S^-1───×───S──────×──────────@──────────@──────────────S^-1───────S^-1───@───S^-1───X───
              │                     │                                           │
3: ────H──────×───S──────────S──────@──────────@───H──────────S──────×──────────@───S──────@───
                                               │                     │                     │
4: ────S──────X───H──────────S──────@───H──────@──────────×───H──────×───H──────@───H──────@───
              │                     │                     │                     │
5: ────S^-1───@──────────×───S^-1───@──────────────S^-1───×──────────×───H──────@

# Entropy calculation by RT formula

In [197]:
start_time = time.perf_counter()


# obtain stabilizers 
stabs = stabilizers_through_clifford_plus_T(circ)
qubits = list(circ.all_qubits())

# subregion A
A = qubits[:13]      

# logical operators supported on A
logics_A = logical_operators_supported_on_A(stabs, qubits, A)

# group logical operators into anti-commuting pairs and singlet
pairs_A, singlets_A = canonicalize_cirq_generators(logics_A, qubits)

# obtain bulk operator by logical_tomography
rho_bulk = logical_tomography(circ, pairs_A, singlets_A)

# stabilizer generators supported on A
stabs_A = stabilizers_supported_on_A(stabs,qubits,A)

# obtain area operator
area_operator = len(A)-len(stabs_A)-len(pairs_A)-len(singlets_A)

# entropy calculation
S_bulk = entropy(rho_bulk)
SA = area_operator + S_bulk

end_time = time.perf_counter()


print('bulk entropy is ',S_bulk, ',   area operator is ',area_operator)
print('S(A) = ',SA)
print('time cost is ', end_time - start_time)


bulk entropy is  0.9999999999999999 ,   area operator is  0
S(A) =  0.9999999999999999
time cost is  3.779069917043671


# Brute-force calculation of entropy

In [4]:
qubits = list(circ.all_qubits())
A = qubits[:13]   
start_time = time.perf_counter()

rho_A = reduced_density_matrix_from_circuit(circ,A)
print('entropy is, ', entropy(rho_A))

end_time = time.perf_counter()

print('time cost is ', start_time - end_time)


/Users/grant/Nutstore Files/Nutstore/Clifford T/.venv/lib/python3.14/site-packages/cirq/sim/state_vector_simulator.py:134: UserWarning: final state vector's norm=np.float32(1.0070364) is too far from 1, 0.007036447525024414 > 0.0003452669770922512.skipping renormalization
  warnings.warn(


entropy is,  1.0062421560287476
time cost is  -285.5833174169529
